The Blood-brain barrier penetration (BBBP)
--------------------------------------------------------------

### All libraries we need

In [45]:
import os
import os.path as osp
import shutil
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import copy
import time


import torch
import torch.nn as nn
from torch.optim import Adam


#********************************************************#
'''
load_dataset contain lots of functions for loading several datasets and 
also there is a function as name get_ dataloader for generating a
dictionary of training, validation, and testing dataLoader.
'''
from load_dataset import get_dataset, get_dataloader

#********************************************************#
'''
As we need several arguments for training process, we store all argument in configure file. 
For using this file, you need the library'Typed Argument Parser (Tap). So you need 'pip install typed-argument-parser'. 
'''
from Configures import data_args, train_args, model_args


from torch.nn.utils.prune import global_unstructured, L1Unstructured

### start loading data

In [28]:
print(data_args.dataset_name)
print(data_args.dataset_dir)



bbbp
/datasets


In [29]:
dataset = get_dataset(data_args.dataset_dir, data_args.dataset_name)
input_dim = dataset.num_node_features
output_dim = int(dataset.num_classes)


print(input_dim)
print(output_dim)

9
2


C:\Users\Dell\AppData\Local\Programs\Python\Python310\lib\site-packages\torch_geometric\data\in_memory_dataset.py:183: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


### Data Analysis

In [30]:
avg_nodes = 0.0
avg_edge_index = 0.0
for i in range(len(dataset)):
    avg_nodes += dataset[i].x.shape[0]
    avg_edge_index += dataset[i].edge_index.shape[1]
avg_nodes /= len(dataset)
avg_edge_index /= len(dataset)
print(f"graphs {len(dataset)}, avg_nodes{avg_nodes :.4f}, avg_edge_index_{avg_edge_index/2 :.4f}")

best_acc = 0.0
data_size = len(dataset)
print(f'The total num of dataset is {data_size}')



graphs 2050, avg_nodes23.9356, avg_edge_index_25.8151
The total num of dataset is 2050


In [31]:


# Read the CSV file
df = pd.read_csv('datasets/bbbp/raw/BBBP.csv')

# Print the shape of the dataset
print("The shape of the dataset is:", df.shape)

# Print the columns of the dataset
print("The columns of the dataset are:", df.columns)

# Print the summary statistics of the dataset
print("The summary statistics of the dataset are:")
print(df.describe())

# Print some sample rows of the dataset
print("Some sample rows of the dataset are:")
df.head(5)

The shape of the dataset is: (2050, 4)
The columns of the dataset are: Index(['num', 'name', 'p_np', 'smiles'], dtype='object')
The summary statistics of the dataset are:
               num         p_np
count  2050.000000  2050.000000
mean   1027.376098     0.764390
std     592.836849     0.424483
min       1.000000     0.000000
25%     514.250000     1.000000
50%    1026.500000     1.000000
75%    1540.750000     1.000000
max    2053.000000     1.000000
Some sample rows of the dataset are:


,num,name,p_np,smiles
0,1,Propanolol,1,[Cl].CC(C)NCC(O)COc1cccc2ccccc12
1,2,Terbutylchlorambucil,1,C(=O)(OC(C)(C)C)CCCc1ccc(cc1)N(CCCl)CCCl
2,3,40730,1,c12c3c(N4CCN(C)CC4)c(F)cc1c(c(C(O)=O)cn2C(C)CO...
3,4,24,1,C1CCN(CC1)Cc1cccc(c1)OCCCNC(=O)C
4,5,cloxacillin,1,Cc1onc(c2ccccc2Cl)c1C(=O)N[C@H]3[C@H]4SC(C)(C)...


### Visualizing

In [ ]:
smiles_list = df["smiles"][10:20].tolist()
name_list = df["name"][10:20].tolist()
label = df["p_np"][10:20].tolist()
new_label=[]
for i in range(10): 
        if label[i]==1:
                new_label.append( 'blood-brain barrier' )
        else :   
                new_label.append('non-blood-brain barrier')

                # Convert each sublist in new_label to a string
new_label_str = [' - '.join(sublist) for sublist in new_label]
#new_label=[for i in len(label): 'blood-brain barrier' if label[i]==1 else 'non-blood-brain barrier']

In [ ]:
new_label

In [ ]:
smiles_list = df["smiles"][10:20].tolist()
name_list = df["name"][10:20].tolist()
label = df["p_np"][10:20].tolist()
new_label=[]
for i in range(len(label)): 
        if label[i]==1:
                new_label.append( name_list[i]+'- blood-brain barrier')
        else :   
                new_label.append(name_list[i]+'- non-blood-brain barrier')

In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw

# Read the CSV file
df = pd.read_csv('datasets/bbbp/raw/BBBP.csv')

# Extract the SMILES strings and names of the first 6 compounds
smiles_list = df["smiles"][10:16].tolist()
name_list = df["name"][10:16].tolist()
label = df["p_np"][10:16].tolist()
new_label=[]
for i in range(len(label)): 
        if label[i]==1:
                new_label.append( name_list[i]+'- blood-brain barrier')
        else :   
                new_label.append(name_list[i]+'- non-blood-brain barrier')

	
plt.rcParams['figure.figsize'] = [20, 5] 
plt.rcParams.update({'font.size': 12})
# Convert the SMILES strings into RDKit molecule objects
mol_list = [Chem.MolFromSmiles(smiles) for smiles in smiles_list]

# Create a grid image with 2 rows and 3 columns and put the names as legends
img = Draw.MolsToGridImage(mol_list, molsPerRow=3,subImgSize=(300, 300), legends=new_label)

# img is an IPython.display.Image object, we can get raw PNG data from it
png = img.data

# Write raw PNG data to file
with open("Images/BBBP-sample.png", "wb") as f:
    f.write(png)

In [ ]:
img = Draw.MolsToGridImage(mol_list, molsPerRow=5, subImgSize=(300, 300), legends=new_label, useSVG=False)

# img is an IPython.display.Image object, we can get raw PNG data from it
png = img.data

# Write raw PNG data to file
with open("output.png", "wb") as f:
    f.write(png)

### Preprocessing and cleaning dataset

In [32]:
#cleaned_dataset = [graph for graph in dataset if graph.edge_index.numpy()!=[]]
cleaned_dataset = [graph for graph in dataset if graph.edge_index.numpy().size> 0]
cleaned_dataset_len=len(cleaned_dataset)
print(f'The number of graphs after cleaning dataset is: {cleaned_dataset_len}')

The number of graphs after cleaning dataset is: 2039


In [33]:
dataloader=get_dataloader(cleaned_dataset, batch_size=train_args.batch_size, random_split_flag=True, data_split_ratio=[0.8, 0.1, 0.1], seed=2)

C:\Users\Dell\AppData\Local\Programs\Python\Python310\lib\site-packages\torch_geometric\deprecation.py:22: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


### Traninig Process

In [34]:
model_args.model_name.lower()

'gcn'

In [35]:
from GCN import GCNNet

def get_model(input_dim, output_dim, model_args):
    if model_args.model_name.lower() == 'gcn':
        return GCNNet(input_dim, output_dim, model_args)
    elif model_args.model_name.lower() == 'gat':
        return GATNet(input_dim, output_dim, model_args)
    elif model_args.model_name.lower() == 'gin':
        return GINNet(input_dim, output_dim, model_args)
    else:
        raise NotImplementedError
        


class GnnBase(nn.Module):
    def __init__(self):
        super(GnnBase, self).__init__()

    def forward(self, data):
        data = data.to(self.device)
        logits, prob, emb = self.model(data)
        return logits, prob, emb

    def update_state_dict(self, state_dict):
        original_state_dict = self.state_dict()
        loaded_state_dict = dict()
        for k, v in state_dict.items():
            if k in original_state_dict.keys():
                loaded_state_dict[k] = v
        self.load_state_dict(loaded_state_dict)

    def to_device(self):
        self.to(self.device)

    def save_state_dict(self):
        pass


class GnnNets(GnnBase):
    def __init__(self, input_dim, output_dim, model_args):
        super(GnnNets, self).__init__()
        self.model = get_model(input_dim, output_dim, model_args)
        self.device = model_args.device

    def forward(self, data):
        data = data.to(self.device)
        logits, prob, emb = self.model(data)
        return logits, prob, emb



In [36]:
model = GnnNets(input_dim, output_dim, model_args)
model.to_device()
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=train_args.learning_rate, weight_decay=train_args.weight_decay)

In [11]:
model.model.gnn_layers


ModuleList(
  (0): GCNConv(9, 128)
  (1-2): 2 x GCNConv(128, 128)
)

In [ ]:
#model.features._modules.items()

In [37]:
def evaluate_GC(eval_dataloader,model, criterion):
    acc = []
    loss_list = []
    model.eval()
    with torch.no_grad():
        for batch in eval_dataloader:
            logits, probs, _ = model(batch)
            loss = criterion(logits, batch.y)

            ## record
            _, prediction = torch.max(logits, -1)
            loss_list.append(loss.item())
            acc.append(prediction.eq(batch.y).cpu().numpy())

        eval_state = {'loss': np.average(loss_list),
                      'acc': np.concatenate(acc, axis=0).mean()}

    return eval_state


def test_GC(test_dataloader,model, criterion):
    acc = []
    loss_list = []
    pred_probs = []
    predictions = []
    model.eval()
    with torch.no_grad():
        for batch in test_dataloader:
            logits, probs, _ = model(batch)
            loss = criterion(logits, batch.y)

            # record
            _, prediction = torch.max(logits, -1)
            loss_list.append(loss.item())
            acc.append(prediction.eq(batch.y).cpu().numpy())
            predictions.append(prediction)
            pred_probs.append(probs)

    test_state = {'loss': np.average(loss_list),
                  'acc': np.average(np.concatenate(acc, axis=0).mean())}

    pred_probs = torch.cat(pred_probs, dim=0).cpu().detach().numpy()
    predictions = torch.cat(predictions, dim=0).cpu().detach().numpy()
    return test_state, pred_probs, predictions

def save_best(ckpt_dir, epoch, model, model_name, eval_acc, is_best):
    print('saving....')
    model.to('cpu')
    state = {
        'net': model.state_dict(),
        'epoch': epoch,
        'acc': eval_acc
    }
    pth_name = f"{model_name}_latest.pth"
    best_pth_name = f'{model_name}_best.pth'
    ckpt_path = os.path.join(ckpt_dir, pth_name)
    torch.save(state, ckpt_path)
    if is_best:
        shutil.copy(ckpt_path, os.path.join(ckpt_dir, best_pth_name))
    model.to_device()

### save path for model

In [38]:

if not os.path.isdir('checkpoint'):
    os.mkdir('checkpoint')
if not os.path.isdir(os.path.join('checkpoint', data_args.dataset_name)):
    os.mkdir(os.path.join('checkpoint', f"{data_args.dataset_name}"))
ckpt_dir = f"./checkpoint/{data_args.dataset_name}/"



In [39]:
def train(callbacks = None):
    logits, probs, _ = model(batch)
    loss = criterion(logits, batch.y)

    # optimization
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_value_(model.parameters(), clip_value=2.0)
    optimizer.step()
    
    ## record
    _, prediction = torch.max(logits, -1)
    loss_list.append(loss.item())
    acc.append(prediction.eq(batch.y).cpu().numpy())
    
    if callbacks is not None:
        for callback in callbacks:
                callback()
  
    return acc

#### Training

In [ ]:
best_acc=0
early_stop_count = 0

for epoch in range(20):
  
    acc=[]
    loss_list = []
    model.train()
    for batch in dataloader['train']:
        acc=train(callbacks = None)
    
    # report train msg
    print(f"Train Epoch:{epoch}  |Loss: {np.average(loss_list):.3f} | "
          f"Acc: {np.concatenate(acc, axis=0).mean():.3f}")
    
    # report eval msg
    eval_state = evaluate_GC(dataloader['eval'], model, criterion)
    print(f"Eval Epoch: {epoch} | Loss: {eval_state['loss']:.3f} | Acc: {eval_state['acc']:.3f}")
    
    # only save the best model
    is_best = (eval_state['acc'] > best_acc)

    if eval_state['acc'] > best_acc:
        early_stop_count = 0
    else:
        early_stop_count += 1

    if early_stop_count > train_args.early_stopping:
        break

    if is_best:
        best_acc = eval_state['acc']
        early_stop_count = 0
    if is_best or epoch % train_args.save_epoch == 0:
        save_best(ckpt_dir, epoch, model, model_args.model_name, eval_state['acc'], is_best)

print(f"The best validation accuracy is {best_acc}.")
  


In [ ]:
 print(f"Acc: {np.concatenate(acc, axis=0).mean():.3f}")

### Evaluation 

In [ ]:
best_checkpoint = dict()
best_checkpoint['state_dict'] = copy.deepcopy(model.state_dict())
model.load_state_dict(best_checkpoint['state_dict'])
recover_model = lambda: model.load_state_dict(best_checkpoint['state_dict'])

t0=time.time()
test_state, _, _ = test_GC(dataloader['test'], model, criterion)
base_model_accuracy= test_state['acc']
t1=time.time()
t_base_model=t1 - t0
###
base_model_size = get_model_size(model,count_nonzero_only=True)
num_parm_base_model=get_num_parameters(model, count_nonzero_only=True)
###   
print(f"dense model has accuracy on test set={base_model_accuracy:.2f}%")
print(f"dense model has size={base_model_size/MiB:.2f} MiB")
print(f"The time inference of base model is ={t_base_model}") 
print(f"The number of parametrs of base model is:{num_parm_base_model}")   


# Let's Prune the Model and Re-Evaluate the Accuracy.
## Gobal Pruning

In [ ]:
model.model

In [ ]:
for n, m in model.named_parameters():

            print(n)         

In [40]:
from torch.nn.utils.prune import global_unstructured, L1Unstructured

sparsity = 0.9
parameters_to_prune = [
    (model.model.gnn_layers[0].lin, 'weight'),
    (model.model.gnn_layers[1].lin, 'weight'),
    (model.model.gnn_layers[2].lin, 'weight'),
    (model.model.mlps[0], 'weight')
]

In [46]:
def get_num_parameters(model: nn.Module, count_nonzero_only=False) -> int:
    """
    calculate the total number of parameters of model
    :param count_nonzero_only: only count nonzero weights
    """
    num_counted_elements = 0
    for param in model.parameters():
        if count_nonzero_only:
            num_counted_elements += param.count_nonzero()
        else:
            num_counted_elements += param.numel()
    return num_counted_elements


def get_model_size(model: nn.Module, data_width=32, count_nonzero_only=False) -> int:
    """
    calculate the model size in bits
    :param data_width: #bits per element
    :param count_nonzero_only: only count nonzero weights
    """
    return get_num_parameters(model, count_nonzero_only) * data_width

Byte = 8
KiB = 1024 * Byte
MiB = 1024 * KiB
GiB = 1024 * MiB

In [49]:
def global_L1(model,parameters_to_prune, sparsity):
        global_unstructured(
        parameters_to_prune,
        pruning_method= L1Unstructured,
        amount=sparsity, )  
        
  
        for module in parameters_to_prune:
           prune.remove(module[0],'weight')


def get_model_sparsity(model: nn.Module) -> float:
        Sparsity=dict()
        global_zero=0
        global_nzero=0
        layyers=[]
        spars=[]
        for name, param in model.named_parameters(): 
            if 'weight' in name:
                        zero=float(torch.sum(param == 0))
                        nzero=float(param.nelement())
                        sparsity=  float(zero)/ float(nzero)
                        print( f'Sparsity in {name}: {sparsity:.3f}' )
                        layyers.append(name)
                        spars.append(sparsity)
                        global_zero +=zero
                        global_nzero +=nzero
                        


        Sparsity={key: value for key, value in zip(layyers,spars)}
        global_sparsity= float(global_zero) /float(global_nzero)
        Sparsity.update({'Global sparsity':  global_sparsity})
        print("Global sparsity: {:.3f}".format(global_sparsity))
        return   Sparsity   

In [50]:
global_L1(model,parameters_to_prune,  sparsity)

get_model_sparsity(model)

Sparsity in model.gnn_layers.0.lin.weight: 0.678
Sparsity in model.gnn_layers.1.lin.weight: 0.907
Sparsity in model.gnn_layers.2.lin.weight: 0.907
Sparsity in model.mlps.0.weight: 1.000
Global sparsity: 0.900


{'model.gnn_layers.0.lin.weight': 0.6779513888888888,
 'model.gnn_layers.1.lin.weight': 0.90728759765625,
 'model.gnn_layers.2.lin.weight': 0.90673828125,
 'model.mlps.0.weight': 1.0,
 'Global sparsity': 0.8999882958801498}

In [ ]:
print('_______________________________________________________')
print(f'Prune the Model and Re-Evaluate the Accuracy')

t0=time.time()
test_state, _, _ = test_GC(dataloader['test'], model, criterion)
pruned_model_accuracy= test_state['acc']
t1=time.time()
t_pruned_model=t1 - t0
###
pruned_model_size = get_model_size(model,count_nonzero_only=True)
num_parm_pruned_model=get_num_parameters(model, count_nonzero_only=True)
###   
print(f"{sparsity*100}% sparse model has accuracy on test set={pruned_model_accuracy:.2f}%")
print(f"{sparsity*100}% sparse model has size={pruned_model_size/MiB:.2f} MiB")
print(f"The time inference of {sparsity*100}% sparse model is ={t_pruned_model}") 
print(f"The number of parametrs of {sparsity*100}% sparse model is:{num_parm_pruned_model}")
print(f"{sparsity*100}% sparse model has size={pruned_model_size/MiB:.2f} MiB, "
      f"which is {base_model_size/pruned_model_size:.2f}X smaller than "
      f"the {base_model_size/MiB:.2f} MiB dense model")

## Finetuning Fine-grained Pruned Sparse Model

In [ ]:
print('_______________________________________________________')
print(f'Finetuning Fine-grained Pruned Sparse Model')

best_sparse_checkpoint = dict()
best_sparse_acc = 0
num_finetune_epochs=100

 
for epoch in range(num_finetune_epochs):
    # At the end of each train iteration, we have to apply the pruning mask
    #    to keep the model sparse during the training
    acc=[]
    loss_list = []

    for batch in dataloader['train']:
        acc=train(callbacks=[lambda: global_L1(model,parameters_to_prune, sparsity)])        

    eval_state = evaluate_GC(dataloader['eval'], model, criterion)
    acc_eval=eval_state['acc']


    is_best = acc_eval > best_sparse_acc
    if is_best:
        best_sparse_checkpoint['state_dict'] = copy.deepcopy(model.state_dict())
        best_sparse_acc = acc_eval

    if epoch % 20 == 0:
         print(f'Epoch {epoch} Sparse Accuracy {acc_eval:.2f}% / Best Sparse Accuracy: {best_sparse_acc:.2f}%')

   

model.load_state_dict(best_sparse_checkpoint['state_dict'])

t0=time.time()
test_state, _, _ = test_GC(dataloader['test'], model, criterion)
pruned_finetune_model_accuracy= test_state['acc']
t1=time.time()
t_pruned_finetune_model=t1 - t0
###
pruned_finetune_model_size = get_model_size(model,count_nonzero_only=True)
num_parm_pruned_finetune_model=get_num_parameters(model, count_nonzero_only=True)
###   
print(f"{sparsity*100}% sparse model has accuracy on test set={pruned_finetune_model_accuracy:.2f}%")
print(f"{sparsity*100}% sparse model has size={pruned_finetune_model_size/MiB:.2f} MiB")
print(f"The time inference of {sparsity*100}% sparse model is ={t_pruned_finetune_model}") 
print(f"The number of parametrs of {sparsity*100}% sparse model is:{num_parm_pruned_finetune_model}")
print(f"{sparsity*100}% sparse model has size={pruned_finetune_model_size/MiB:.2f} MiB, "
      f"which is {base_model_size/pruned_finetune_model_size:.2f}X smaller than "
  f"the {base_model_size/MiB:.2f} MiB dense model")


## Manual Measurement

In [25]:
import statistics as stat

sparsity=0.9
Eva_final=dict()

Base_model_accuracy=[]
T_base_model=[]
Num_parm_base_model=[]
Base_model_size=[]

Pruned_model_accuracy=[]
T_pruned_model=[]
Num_parm_pruned_model=[]
Pruned_model_size=[]

Pruned_finetune_model_accuracy=[]
T_pruned_finetune_model=[]
Num_parm_pruned_finetune_model=[]
Pruned_finetune_model_size=[]

Spar_gnn_layers_0_lin_w=[]
Spar_gnn_layers_1_lin_w=[]
Spar_gnn_layers_2_lin_w=[]
Spar_mlps_0=[]
Global_spar=[]          

In [56]:
num_epoch=100
Eva=dict()

print(f'Training and evaluation before pruning ')
model = GnnNets(input_dim, output_dim, model_args)
model.to_device()
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=train_args.learning_rate, weight_decay=train_args.weight_decay)
    
for epoch in range(num_epoch):  
    acc=[]
    loss_list = []
    model.train()
    for batch in dataloader['train']:
        acc=train(callbacks=None)        

    eval_state = evaluate_GC(dataloader['eval'], model, criterion)
    accuracy=eval_state['acc']

    if epoch % 20 == 0:   
        # report train msg
        print(f"Train Epoch:{epoch}  |Loss: {np.average(loss_list):.3f} | "
              f"Acc: {np.concatenate(acc, axis=0).mean():.3f}")
        print(f"Eval Epoch: {epoch} | Loss: {eval_state['loss']:.3f} | Acc: {accuracy:.3f}")
    

best_checkpoint = dict()
best_checkpoint['state_dict'] = copy.deepcopy(model.state_dict())
model.load_state_dict(best_checkpoint['state_dict'])
recover_model = lambda: model.load_state_dict(best_checkpoint['state_dict'])

t0=time.time()
test_state, _, _ = test_GC(dataloader['test'], model, criterion)
base_model_accuracy= test_state['acc']
t1=time.time()
t_base_model=t1 - t0
###
base_model_size = get_model_size(model,count_nonzero_only=True)
num_parm_base_model=get_num_parameters(model, count_nonzero_only=True)
###   
print(f"dense model has accuracy on test set={base_model_accuracy:.2f}%")
print(f"dense model has size={base_model_size/MiB:.2f} MiB")
print(f"The time inference of base model is ={t_base_model}") 
print(f"The number of parametrs of base model is:{num_parm_base_model}")   

#Update my Eva dictionary
Eva.update({'base model accuracy': base_model_accuracy,
            'time inference of base model': t_base_model,
            'number parmameters of base model': num_parm_base_model,
            'size of base model': base_model_size})


print('_______________________________________________________')
print(f'Prune the Model and Re-Evaluate the Accuracy')
recover_model()
parameters_to_prune = [
    (model.model.gnn_layers[0].lin, 'weight'),
    (model.model.gnn_layers[1].lin, 'weight'),
    (model.model.gnn_layers[2].lin, 'weight'),
    (model.model.mlps[0], 'weight')
]            
global_L1(model,parameters_to_prune,  sparsity)

###Sparsities of layyer
spar_dict=get_model_sparsity(model)  
Eva.update(spar_dict)



# Result
t0=time.time()
test_state, _, _ = test_GC(dataloader['test'], model, criterion)
pruned_model_accuracy= test_state['acc']
t1=time.time()
t_pruned_model=t1 - t0
###
pruned_model_size = get_model_size(model,count_nonzero_only=True)
num_parm_pruned_model=get_num_parameters(model, count_nonzero_only=True)
###   
print(f"{sparsity*100}% sparse model has accuracy on test set={pruned_model_accuracy:.2f}%")
print(f"{sparsity*100}% sparse model has size={pruned_model_size/MiB:.2f} MiB")
print(f"The time inference of {sparsity*100}% sparse model is ={t_pruned_model}") 
print(f"The number of parametrs of {sparsity*100}% sparse model is:{num_parm_pruned_model}")
print(f"{sparsity*100}% sparse model has size={pruned_model_size/MiB:.2f} MiB, "
      f"which is {base_model_size/pruned_model_size:.2f}X smaller than "
      f"the {base_model_size/MiB:.2f} MiB dense model")



#Update my Eva dictionary
Eva.update({'pruned model accuracy': pruned_model_accuracy,
            'time inference of pruned model': t_pruned_model,
            'number parmameters of pruned model': num_parm_pruned_model,
            'size of pruned model': pruned_model_size})



print('_______________________________________________________')
print(f'Finetuning Fine-grained Pruned Sparse Model')

best_sparse_checkpoint = dict()
best_sparse_acc = 0
num_finetune_epochs=100

 
for epoch in range(num_finetune_epochs):
    # At the end of each train iteration, we have to apply the pruning mask
    #    to keep the model sparse during the training
    acc=[]
    loss_list = []

    for batch in dataloader['train']:
        acc=train(callbacks=[lambda: global_L1(model,parameters_to_prune, sparsity)])        

    eval_state = evaluate_GC(dataloader['eval'], model, criterion)
    acc_eval=eval_state['acc']


    is_best = acc_eval > best_sparse_acc
    if is_best:
        best_sparse_checkpoint['state_dict'] = copy.deepcopy(model.state_dict())
        best_sparse_acc = acc_eval

    if epoch % 20 == 0:
         print(f'Epoch {epoch} Sparse Accuracy {acc_eval:.2f}% / Best Sparse Accuracy: {best_sparse_acc:.2f}%')

   
model.load_state_dict(best_sparse_checkpoint['state_dict'])

t0=time.time()
test_state, _, _ = test_GC(dataloader['test'], model, criterion)
pruned_finetune_model_accuracy= test_state['acc']
t1=time.time()
t_pruned_finetune_model=t1 - t0
###
pruned_finetune_model_size = get_model_size(model,count_nonzero_only=True)
num_parm_pruned_finetune_model=get_num_parameters(model, count_nonzero_only=True)
###   
print(f"{sparsity*100}% sparse model has accuracy on test set={pruned_finetune_model_accuracy:.2f}%")
print(f"{sparsity*100}% sparse model has size={pruned_finetune_model_size/MiB:.2f} MiB")
print(f"The time inference of {sparsity*100}% sparse model is ={t_pruned_finetune_model}") 
print(f"The number of parametrs of {sparsity*100}% sparse model is:{num_parm_pruned_finetune_model}")
print(f"{sparsity*100}% sparse model has size={pruned_finetune_model_size/MiB:.2f} MiB, "
      f"which is {base_model_size/pruned_finetune_model_size:.2f}X smaller than "
  f"the {base_model_size/MiB:.2f} MiB dense model")


 #Update my Eva dictionary
Eva.update({'pruned and finetune model accuracy': pruned_finetune_model_accuracy,
            'time inference of pruned and finetune model': t_pruned_finetune_model,
            'number parmameters of pruned and finetune model': num_parm_pruned_finetune_model,
            'size of pruned and finetune model':  pruned_finetune_model_size})



Base_model_accuracy.append(Eva['base model accuracy'])
T_base_model.append(Eva['time inference of base model'])
Num_parm_base_model.append(int(Eva['number parmameters of base model']))
Base_model_size.append(int(Eva['size of base model']))

Pruned_model_accuracy.append(Eva['pruned model accuracy'])
T_pruned_model.append(Eva['time inference of pruned model'])
Num_parm_pruned_model.append(int(Eva['number parmameters of pruned model']))
Pruned_model_size.append(int(Eva['size of pruned model']))

Pruned_finetune_model_accuracy.append(Eva['pruned and finetune model accuracy'])
T_pruned_finetune_model.append(Eva['time inference of pruned and finetune model'])
Num_parm_pruned_finetune_model.append(int(Eva['number parmameters of pruned and finetune model']))
Pruned_finetune_model_size.append(int(Eva['size of pruned and finetune model']))

Spar_gnn_layers_0_lin_w.append(Eva['model.gnn_layers.0.lin.weight'])
Spar_gnn_layers_1_lin_w.append(Eva['model.gnn_layers.1.lin.weight'])
Spar_gnn_layers_2_lin_w.append(Eva['model.gnn_layers.2.lin.weight'])
Spar_mlps_0.append(Eva['model.mlps.0.weight'])
Global_spar.append(Eva['Global sparsity'])    



Training and evaluation before pruning 
Train Epoch:0  |Loss: 0.602 | Acc: 0.748
Eval Epoch: 0 | Loss: 0.569 | Acc: 0.754
Train Epoch:20  |Loss: 0.441 | Acc: 0.812
Eval Epoch: 20 | Loss: 0.462 | Acc: 0.818
Train Epoch:40  |Loss: 0.384 | Acc: 0.842
Eval Epoch: 40 | Loss: 0.433 | Acc: 0.837
Train Epoch:60  |Loss: 0.343 | Acc: 0.863
Eval Epoch: 60 | Loss: 0.350 | Acc: 0.847
Train Epoch:80  |Loss: 0.290 | Acc: 0.890
Eval Epoch: 80 | Loss: 0.354 | Acc: 0.852
dense model has accuracy on test set=0.83%
dense model has size=0.13 MiB
The time inference of base model is =0.1149294376373291
The number of parametrs of base model is:34526
_______________________________________________________
Prune the Model and Re-Evaluate the Accuracy
Sparsity in model.gnn_layers.0.lin.weight: 0.608
Sparsity in model.gnn_layers.1.lin.weight: 0.912
Sparsity in model.gnn_layers.2.lin.weight: 0.911
Sparsity in model.mlps.0.weight: 0.793
Global sparsity: 0.900
90.0% sparse model has accuracy on test set=0.35%
90.0% 

In [58]:
Eva_final=dict()
base_model_accuracy_mean = stat.mean(Base_model_accuracy)
base_model_accuracy_std =  stat.stdev(Base_model_accuracy)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)

Eva_final.update({'base model accuracy':float(format(base_model_accuracy_mean, '.3f'))})
Eva_final.update({'Std of base model accuracy':float(format(base_model_accuracy_std, '.3f'))})
                 
t_base_model_mean =stat.mean(T_base_model)
t_base_model_std =stat.stdev(T_base_model)  
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'time inference of base model':float(format(t_base_model_mean, '.3f'))})
Eva_final.update({'Std of time inference of base model':float(format(t_base_model_std, '.3f'))})


num_parm_base_model_mean = stat.mean(Num_parm_base_model)
num_parm_base_model_std = stat.stdev(Num_parm_base_model)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'number parmameters of base model':num_parm_base_model_mean})
Eva_final.update({'Std of number parmameters of base model':num_parm_base_model_std})

base_model_size_mean = stat.mean(Base_model_size)
base_model_size_std = stat.stdev(Base_model_size)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'base_model_size':base_model_size_mean})
Eva_final.update({'Std of base_model_size':base_model_size_std})

#################################

pruned_model_accuracy_mean =stat.mean(Pruned_model_accuracy)
pruned_model_accuracy_std = stat.stdev(Pruned_model_accuracy)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'pruned model accuracy':float(format(pruned_model_accuracy_mean, '.3f'))})
Eva_final.update({'Std of pruned model accuracy':float(format(pruned_model_accuracy_std, '.3f'))})
                 

t_pruned_model_mean = stat.mean(T_pruned_model)
t_pruned_model_std =stat.stdev(T_pruned_model)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'time inference of pruned model':float(format(t_pruned_model_mean, '.3f'))})
Eva_final.update({'Std of time inference of pruned model':float(format(t_pruned_model_std, '.3f'))})

num_parm_pruned_model_mean = stat.mean(Num_parm_pruned_model)
num_parm_pruned_model_std = stat.stdev(Num_parm_pruned_model)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'number parmameters of pruned model':num_parm_pruned_model_mean})
Eva_final.update({'Std of number parmameters of pruned model':num_parm_pruned_model_std})

pruned_model_size_mean =stat.mean( Pruned_model_size)
pruned_model_size_std = stat.stdev(Pruned_model_size)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'pruned model size':pruned_model_size_mean})
Eva_final.update({'Std of pruned_model_size':pruned_model_size_std })

#################################
pruned_finetune_model_accuracy_mean =stat.mean(Pruned_finetune_model_accuracy)
pruned_finetune_model_accuracy_std = stat.stdev(Pruned_finetune_model_accuracy)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'pruned finetune model accuracy':float(format(pruned_finetune_model_accuracy_mean, '.3f'))})
Eva_final.update({'Std of pruned finetune model accuracy':float(format(pruned_finetune_model_accuracy_std, '.3f'))})                 

t_pruned_finetune_model_mean =stat.mean(T_pruned_finetune_model)
t_pruned_finetune_model_std =stat.stdev(T_pruned_finetune_model)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'time inference of pruned finetune model':float(format(t_pruned_finetune_model_mean,'.3f'))})
Eva_final.update({'Std of time inference of pruned finetune model':float(format(t_pruned_finetune_model_std,'.3f'))})

num_parm_pruned_finetune_model_mean =stat.mean(Num_parm_pruned_finetune_model)
num_parm_pruned_finetune_model_std = stat.stdev(Num_parm_pruned_finetune_model)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'number parmameters of pruned finetune model':num_parm_pruned_finetune_model_mean})
Eva_final.update({'Std of number parmameters of pruned finetune model':num_parm_pruned_finetune_model_std })

pruned_finetune_model_size_mean = stat.mean(Pruned_finetune_model_size)
pruned_finetune_model_size_std = stat.stdev(Pruned_finetune_model_size)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'pruned finetune model size':pruned_finetune_model_size_mean})
Eva_final.update({'Std of pruned finetune model size':pruned_finetune_model_size_std})



sparsity_gnn_layers_0_lin_w_mean = stat.mean(Spar_gnn_layers_0_lin_w)
sparsity_gnn_layers_0_lin_w_std = stat.stdev(Spar_gnn_layers_0_lin_w)
Eva_final.update({'Sparsity in gnn_layers[0].lin.weight':float(format(sparsity_gnn_layers_0_lin_w_mean,'.3f'))})
Eva_final.update({'Std of Sparsity in gnn_layers[0].lin.weight':float(format(sparsity_gnn_layers_0_lin_w_std,'.3f'))})

sparsity_gnn_layers_1_lin_w_mean = stat.mean(Spar_gnn_layers_1_lin_w)
sparsity_gnn_layers_1_lin_w_std = stat.stdev(Spar_gnn_layers_1_lin_w)
Eva_final.update({'Sparsity in gnn_layers[1].lin.weight':float(format(sparsity_gnn_layers_1_lin_w_mean,'.3f'))})
Eva_final.update({'Std of Sparsity in gnn_layers[1].lin.weight':float(format(sparsity_gnn_layers_1_lin_w_std,'.3f'))})


sparsity_gnn_layers_2_lin_w_mean = stat.mean(Spar_gnn_layers_2_lin_w)
sparsity_gnn_layers_2_lin_w_std = stat.stdev(Spar_gnn_layers_2_lin_w)
Eva_final.update({'Sparsity in gnn_layers[2].lin.weight':float(format(sparsity_gnn_layers_2_lin_w_mean,'.3f'))}) 
Eva_final.update({'Std of Sparsity in gnn_layers[2].lin.weight':float(format(sparsity_gnn_layers_2_lin_w_std,'.3f'))}) 

sparsity_mlps_0_mean = stat.mean(Spar_mlps_0)
sparsity_mlps_0_std = stat.stdev(Spar_mlps_0)
Eva_final.update({'Sparsity in gmlps[0].weight':float(format(sparsity_mlps_0_mean,'.3f')) })
Eva_final.update({'Std of Sparsity in gmlps[0].weight':float(format(sparsity_mlps_0_std,'.3f')) })


Global_sparsity_mean = stat.mean(Global_spar)
Global_sparsity_std = stat.stdev(Global_spar)
Eva_final.update({'Global sparsity':float(format(Global_sparsity_mean,'.3f'))})
Eva_final.update({'Std of Global sparsity':float(format(Global_sparsity_std,'.3f'))})


#################################


print(f"All measurement about pruning process of sparsity:{sparsity*100}% ")   
Eva_final

All measurement about pruning process of sparsity:90.0% 


{'base model accuracy': 0.827,
 'Std of base model accuracy': 0.011,
 'time inference of base model': 0.098,
 'Std of time inference of base model': 0.01,
 'number parmameters of base model': 34527.6,
 'Std of number parmameters of base model': 5.683308895353129,
 'base_model_size': 1104883.2,
 'Std of base_model_size': 181.86588465130012,
 'pruned model accuracy': 0.482,
 'Std of pruned model accuracy': 0.199,
 'time inference of pruned model': 0.102,
 'Std of time inference of pruned model': 0.013,
 'number parmameters of pruned model': 3769.6,
 'Std of number parmameters of pruned model': 5.683308895353129,
 'pruned model size': 120627.2,
 'Std of pruned_model_size': 181.86588465130012,
 'pruned finetune model accuracy': 0.849,
 'Std of pruned finetune model accuracy': 0.015,
 'time inference of pruned finetune model': 0.097,
 'Std of time inference of pruned finetune model': 0.005,
 'number parmameters of pruned finetune model': 3778.4,
 'Std of number parmameters of pruned finetun

In [59]:
BBBP_09={'base model accuracy': 0.827,
 'Std of base model accuracy': 0.011,
 'time inference of base model': 0.098,
 'Std of time inference of base model': 0.01,
 'number parmameters of base model': 34527.6,
 'Std of number parmameters of base model': 5.683308895353129,
 'base_model_size': 1104883.2,
 'Std of base_model_size': 181.86588465130012,
 'pruned model accuracy': 0.482,
 'Std of pruned model accuracy': 0.199,
 'time inference of pruned model': 0.102,
 'Std of time inference of pruned model': 0.013,
 'number parmameters of pruned model': 3769.6,
 'Std of number parmameters of pruned model': 5.683308895353129,
 'pruned model size': 120627.2,
 'Std of pruned_model_size': 181.86588465130012,
 'pruned finetune model accuracy': 0.849,
 'Std of pruned finetune model accuracy': 0.015,
 'time inference of pruned finetune model': 0.097,
 'Std of time inference of pruned finetune model': 0.005,
 'number parmameters of pruned finetune model': 3778.4,
 'Std of number parmameters of pruned finetune model': 3.7815340802378077,
 'pruned finetune model size': 120908.8,
 'Std of pruned finetune model size': 121.00909056760985,
 'Sparsity in gnn_layers[0].lin.weight': 0.603,
 'Std of Sparsity in gnn_layers[0].lin.weight': 0.016,
 'Sparsity in gnn_layers[1].lin.weight': 0.903,
 'Std of Sparsity in gnn_layers[1].lin.weight': 0.007,
 'Sparsity in gnn_layers[2].lin.weight': 0.919,
 'Std of Sparsity in gnn_layers[2].lin.weight': 0.007,
 'Sparsity in gmlps[0].weight': 0.838,
 'Std of Sparsity in gmlps[0].weight': 0.038,
 'Global sparsity': 0.9,
 'Std of Global sparsity': 0.0}